# Generate the missing report figures (Google Colab)

Use a **GPU runtime**. This notebook restores the existing ESD/attack artifacts, generates the matched original Stable Diffusion arm, scores all three arms with NudeNet, and downloads only masked report assets. Raw generated images remain private in the Colab runtime.

Before running, put `nudity_n20_sample2024_run0_eval_bundle.tar.gz` in Google Drive, or leave `ARTIFACT_ARCHIVE` empty to upload it when prompted.

In [ ]:
REPO_URL = "https://github.com/Noridom1/Diffusion-MU-Attack.git"
BRANCH = "main"
ARTIFACT_ARCHIVE = ""  # Example: /content/drive/MyDrive/nudity_n20_sample2024_run0_eval_bundle.tar.gz
RUN_FULL_EXPERIMENT = False  # True reruns ESD baseline + attack if you do not have the artifact bundle.

In [ ]:
import os, pathlib, shutil, subprocess

assert shutil.which("nvidia-smi"), "Select Runtime > Change runtime type > GPU, then reconnect."
subprocess.run(["nvidia-smi"], check=True)
repo = pathlib.Path("/content/Diffusion-MU-Attack")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(repo)], check=True)
os.chdir(repo)

## Restore existing ESD and attack artifacts

Skip the upload only when `RUN_FULL_EXPERIMENT=True`. The archive is not stored in public Git because it contains uncensored research outputs.

In [ ]:
import tarfile, tempfile, zipfile

def safe_unpack(archive, destination):
    archive, destination = pathlib.Path(archive), pathlib.Path(destination).resolve()
    if tarfile.is_tarfile(archive):
        with tarfile.open(archive) as bundle:
            for member in bundle.getmembers():
                target = (destination / member.name).resolve()
                assert target == destination or destination in target.parents, f"Unsafe archive path: {member.name}"
            bundle.extractall(destination, filter="data")
    elif zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as bundle:
            for name in bundle.namelist():
                target = (destination / name).resolve()
                assert target == destination or destination in target.parents, f"Unsafe archive path: {name}"
            bundle.extractall(destination)
    else:
        raise ValueError(f"Unsupported archive: {archive}")

if not RUN_FULL_EXPERIMENT:
    archive = ARTIFACT_ARCHIVE
    if archive.startswith("/content/drive/") and not pathlib.Path(archive).exists():
        from google.colab import drive
        drive.mount("/content/drive")
    if not archive:
        from google.colab import files
        archive = next(iter(files.upload()))
    with tempfile.TemporaryDirectory() as tmp:
        safe_unpack(pathlib.Path(archive).expanduser(), tmp)
        matches = list(pathlib.Path(tmp).glob("**/files/results/nudity_n20_sample2024_run0"))
        assert len(matches) == 1, f"Expected one experiment in the archive, found {len(matches)}"
        files_root = matches[0].parents[1]
        shutil.copytree(files_root, repo / "files", dirs_exist_ok=True)
    for arm in ("no_attack", "unlearndiff"):
        assert (repo / "files/results/nudity_n20_sample2024_run0" / arm).is_dir(), f"Missing {arm}"
    print("Existing ESD and attack artifacts restored.")

## Install the pinned environment and model files

This is the longest setup cell. It installs Miniconda, creates the repository's pinned environment, and downloads Stable Diffusion v1.4 plus the ESD checkpoint. If Hugging Face requests authentication, run `huggingface-cli login` and retry.

In [ ]:
miniconda = pathlib.Path("/content/miniconda")
if not (miniconda / "bin/conda").exists():
    installer = pathlib.Path("/content/miniconda.sh")
    subprocess.run(["wget", "-q", "-O", str(installer), "https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh"], check=True)
    subprocess.run(["bash", str(installer), "-b", "-p", str(miniconda)], check=True)
os.environ["PATH"] = f"{miniconda / 'bin'}:{os.environ['PATH']}"
env = os.environ.copy()
env.update({"GPU0": "0", "GPU1": "", "CASE_LIST": "", "N_PROMPTS": "20", "SAMPLE_SEED": "2024", "RUN_SEED": "0"})
subprocess.run(["bash", "scripts/nudity30_linux.sh", "setup"], env=env, check=True)

In [ ]:
if RUN_FULL_EXPERIMENT:
    for command in ("prepare", "baseline", "attack", "evaluate"):
        subprocess.run(["bash", "scripts/nudity30_linux.sh", command], env=env, check=True)
subprocess.run(["bash", "scripts/nudity30_linux.sh", "missing-figures"], env=env, check=True)

## Review and download masked outputs

The preview reads only the automatically masked copies. Inspect every selected image before including it in the submitted report.

In [ ]:
import re
from IPython.display import Image, display

selected = []
for fragment in (repo / "report/figures/erasure_pairs.tex", repo / "report/figures/attack_triplets.tex"):
    selected.extend(re.findall(r"\\includegraphics.*?\{(figures/[^}]+)\}", fragment.read_text()))
for relative in dict.fromkeys(selected):
    path = repo / "report" / relative
    print(relative)
    display(Image(filename=str(path), width=300))

result_root = repo / "files/results/nudity_n20_sample2024_run0"
bundle = result_root / "report_artifacts.tar.gz"
checksum = bundle.with_suffix(bundle.suffix + ".sha256")
assert bundle.is_file() and checksum.is_file()
print(checksum.read_text())

zip_path = result_root / "report_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as output:
    for path in (repo / "report/figures/generated").glob("*.png"):
        output.write(path, path.relative_to(repo))
    for path in [
        repo / "report/figures/erasure_pairs.tex",
        repo / "report/figures/attack_triplets.tex",
        repo / "report/figures/detection_rates.tex",
        repo / "report/figures/report_metrics.json",
        result_root / "original_sd/prompts.csv",
    ]:
        output.write(path, path.relative_to(repo))
print(f"Created {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MiB)")

In [ ]:
from google.colab import files
files.download(str(zip_path))